In [10]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load the messy data
df = pd.read_csv("bike_data.csv")

# ==========================================
# PHASE 1: DIAGNOSE & INSPECT THE DATA
# ==========================================
print("--- 1. DATA INFO (Check Data Types & NaNs) ---")

df.info()

print("\n--- 2. SUMMARY STATISTICS (Check for Outliers) ---")

print(df[['temp', 'humidity', 'windspeed', 'count']].describe())

print("\n--- 3. CATEGORY CHECKS (Check for Invalid Entries) ---")
print("Weather categories:\n", df['weather'].value_counts(dropna=False))
print("\nSeason categories:\n", df['season'].value_counts(dropna=False))

print(f"\n--- 4. DUPLICATES: {df.duplicated().sum()} found ---")


# ==========================================
# PHASE 2: CLEAN & FIX THE DATA
# ==========================================
print("\nInitiating data cleaning pipeline...\n")

# 1. Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()

# 2. Fix Mixed Data Types
# Replace the text strings with integers, then force the whole column to numeric
df_clean['weather'] = df_clean['weather'].replace({'Sunny': 1, 'Cloudy': 2})
df_clean['weather'] = pd.to_numeric(df_clean['weather'], errors='coerce')

# 3. Fix Invalid Categories
# Filter the dataframe to only include valid seasons (1, 2, 3, 4)
df_clean = df_clean[df_clean['season'].isin([1, 2, 3, 4])]

# 4. Handle Outliers and Impossible Values
# Replace sensor errors with NaN so we can impute them in the next step
df_clean.loc[df_clean['windspeed'] > 100, 'windspeed'] = np.nan   
df_clean.loc[df_clean['humidity'] < 0, 'humidity'] = np.nan       
df_clean.loc[df_clean['temp'] > 60, 'temp'] = np.nan              

# 5. Handle Missing Values (NaNs)
# Rule: Never impute your target variable. Drop rows where 'count' is missing.
df_clean = df_clean.dropna(subset=['count'])

# Rule: Fill missing environmental features with the median of their respective columns.
for col in ['temp', 'humidity', 'windspeed']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 6. Final Type Casting
# Ensure discrete categories and the target variable are clean integers
df_clean['count'] = df_clean['count'].astype(int)
df_clean['weather'] = df_clean['weather'].astype(int)

# ==========================================
# RESULTS
# ==========================================
print(f"Original shape: {df.shape}")
print(f"Cleaned shape:  {df_clean.shape}")
print("\n--- CLEAN DATA INFO ---")
df_clean.info()

# Save the clean dataset to use for your statsmodels regression
df_clean.to_csv("clean_bike_data.csv", index=False)

--- 1. DATA INFO (Check Data Types & NaNs) ---
<class 'pandas.DataFrame'>
RangeIndex: 101000 entries, 0 to 100999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   datetime    101000 non-null  str    
 1   season      101000 non-null  int64  
 2   holiday     101000 non-null  int64  
 3   workingday  101000 non-null  int64  
 4   weather     101000 non-null  str    
 5   temp        95991 non-null   float64
 6   humidity    95981 non-null   float64
 7   windspeed   101000 non-null  float64
 8   count       98982 non-null   float64
dtypes: float64(4), int64(3), str(2)
memory usage: 8.9 MB

--- 2. SUMMARY STATISTICS (Check for Outliers) ---
               temp      humidity      windspeed         count
count  95991.000000  95981.000000  101000.000000  98982.000000
mean      21.358771     54.223572      24.821873     89.470904
std       15.840444     21.661527      98.385072     71.241018
min        5.000934    -25.000

In [13]:
# ==============================================================================
# 2. HYPOTHESIS TESTS & 95% CONFIDENCE INTERVALS (Phase 1)
# ==============================================================================
print("=" * 70)
print("SECTION A: HYPOTHESIS TESTING & CONFIDENCE INTERVALS")
print("=" * 70)

# --- 1. T-TEST: Workingday vs Non-Workingday ---
work = df_clean[df_clean['workingday'] == 1]['count']
non_work = df_clean[df_clean['workingday'] == 0]['count']

t_stat, p_val_ttest = stats.ttest_ind(work, non_work)
ci_ttest = stats.ttest_ind(work, non_work).confidence_interval(confidence_level=0.95)

print(f"\n1. Two-Sample T-Test (Workingday vs Non-Workingday):")
print(f"   - T-statistic:       {t_stat:.4f}")
print(f"   - p-value:           {p_val_ttest:.4e}")
print(f"   - Difference 95% CI: [{ci_ttest.low:.2f} to {ci_ttest.high:.2f}] bikes")

# --- 2. ANOVA: Season vs Count ---
season_groups = [group['count'].values for _, group in df_clean.groupby('season')]
f_stat_season, p_val_season = stats.f_oneway(*season_groups)

print(f"\n2. One-Way ANOVA (Season vs Count):")
print(f"   - F-statistic:       {f_stat_season:.4f}")
print(f"   - p-value:           {p_val_season:.4e}")

# Mean & 95% CI per Season
print("   - Season-wise 95% Confidence Intervals:")
for s, group in df_clean.groupby('season')['count']:
    mean = group.mean()
    sem = stats.sem(group)
    ci = stats.t.interval(0.95, len(group) - 1, loc=mean, scale=sem)
    print(f"     * Season {s}: Mean = {mean:.2f} | 95% CI: [{ci[0]:.2f} to {ci[1]:.2f}]")

# --- 3. ANOVA: Weather vs Count ---
weather_groups = [group['count'].values for _, group in df_clean.groupby('weather')]
f_stat_weather, p_val_weather = stats.f_oneway(*weather_groups)

print(f"\n3. One-Way ANOVA (Weather vs Count):")
print(f"   - F-statistic:       {f_stat_weather:.4f}")
print(f"   - p-value:           {p_val_weather:.4e}")

# Mean & 95% CI per Weather Category
print("   - Weather-wise 95% Confidence Intervals:")
for w, group in df_clean.groupby('weather')['count']:
    mean = group.mean()
    sem = stats.sem(group)
    ci = stats.t.interval(0.95, len(group) - 1, loc=mean, scale=sem)
    print(f"     * Weather {w}: Mean = {mean:.2f} | 95% CI: [{ci[0]:.2f} to {ci[1]:.2f}]")


SECTION A: HYPOTHESIS TESTING & CONFIDENCE INTERVALS

1. Two-Sample T-Test (Workingday vs Non-Workingday):
   - T-statistic:       1.3217
   - p-value:           1.8627e-01
   - Difference 95% CI: [-0.32 to 1.64] bikes

2. One-Way ANOVA (Season vs Count):
   - F-statistic:       0.8992
   - p-value:           4.4065e-01
   - Season-wise 95% Confidence Intervals:
     * Season 1: Mean = 89.02 | 95% CI: [88.13 to 89.92]
     * Season 2: Mean = 89.29 | 95% CI: [88.39 to 90.18]
     * Season 3: Mean = 89.35 | 95% CI: [88.45 to 90.25]
     * Season 4: Mean = 90.04 | 95% CI: [89.14 to 90.93]

3. One-Way ANOVA (Weather vs Count):
   - F-statistic:       1583.1328
   - p-value:           0.0000e+00
   - Weather-wise 95% Confidence Intervals:
     * Weather 1: Mean = 100.84 | 95% CI: [100.25 to 101.44]
     * Weather 2: Mean = 78.30 | 95% CI: [77.55 to 79.05]
     * Weather 3: Mean = 56.17 | 95% CI: [54.94 to 57.40]
     * Weather 4: Mean = 36.35 | 95% CI: [33.47 to 39.22]


In [16]:
# ==============================================================================
# 3. MULTIPLE LINEAR REGRESSION & CONFIDENCE INTERVALS (Phase 2)
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION B: REGRESSION MODEL & COEFFICIENT CONFIDENCE INTERVALS")
print("=" * 70)

model_formula = "count ~ temp + humidity + windspeed + C(season) + C(weather) + workingday + holiday"
ols_model = smf.ols(formula=model_formula, data=df_clean).fit()

# Extract parameters
p_values = ols_model.pvalues
coefficients = ols_model.params
conf_intervals = ols_model.conf_int(alpha=0.05)  # 95% confidence interval
conf_intervals.columns = ['CI_Lower', 'CI_Upper']

# Combine into a clean summary table
results_table = pd.DataFrame({
    'Coefficient': coefficients,
    'p-value': p_values,
    '95% CI Lower': conf_intervals['CI_Lower'],
    '95% CI Upper': conf_intervals['CI_Upper'],
    'Significant (p < 0.05)': p_values < 0.05
})

print("\n--- Summary of All Predictors ---")
print(results_table.round(4).to_string())

print("\n--- Model Fit ---")
print(f"R-squared:          {ols_model.rsquared:.4f} ({ols_model.rsquared * 100:.1f}% variance explained)")
print(f"Adjusted R-squared: {ols_model.rsquared_adj:.4f}")


SECTION B: REGRESSION MODEL & COEFFICIENT CONFIDENCE INTERVALS

--- Summary of All Predictors ---
                 Coefficient  p-value  95% CI Lower  95% CI Upper  Significant (p < 0.05)
Intercept            54.8147   0.0000       53.9575       55.6719                    True
C(season)[T.2]        0.1447   0.5685       -0.3527        0.6422                   False
C(season)[T.3]        0.1987   0.4346       -0.2997        0.6970                   False
C(season)[T.4]        0.2432   0.3387       -0.2550        0.7413                   False
C(weather)[T.2]     -23.2514   0.0000      -23.6414      -22.8614                    True
C(weather)[T.3]     -44.2488   0.0000      -44.8930      -43.6045                    True
C(weather)[T.4]     -63.0294   0.0000      -64.8716      -61.1872                    True
temp                  6.8369   0.0000        6.8159        6.8579                    True
humidity             -1.2122   0.0000       -1.2212       -1.2031                    True
w